# Norman 2019 CRISPRa 数据整理与 CellCap 评估

从 GEO GSE133344 下载数据，整理为 AnnData，构造 CellCap 输入，训练并评估组合扰动预测。

In [ ]:
import anndata as ad
import numpy as np
import scanpy as sc
from pathlib import Path

# 加载处理后的 Norman 数据
data_path = Path('../data/processed/norman_crispra.h5ad')
if data_path.exists():
    adata = ad.read_h5ad(data_path)
    print(f'Loaded: {adata.shape}')
    print(f'obs: {list(adata.obs.columns)}')
    print(f'obsm: {list(adata.obsm.keys())}')
else:
    print(f'Data not found: {data_path}')
    print('Run: python scripts/prepare_norman_anndata.py')

In [ ]:
# 扰动条件统计
if 'adata' in locals() and 'condition' in adata.obs.columns:
    counts = adata.obs['condition'].value_counts()
    print(f'Total conditions: {len(counts)}')
    print(f'Control cells: {counts.get("ctrl", 0)}')
    
    single = sum(c for k, c in counts.items() if '+' not in k and k != 'ctrl')
    combo = sum(c for k, c in counts.items() if '+' in k)
    print(f'Single perturbation cells: {single}')
    print(f'Combinatorial perturbation cells: {combo}')
    print()
    print('Top 20 conditions:')
    for cond, cnt in counts.head(20).items():
        print(f'  {cond}: {cnt}')

In [ ]:
# 构造 CellCap 输入并训练
if 'adata' in locals():
    X = adata.layers.get('counts', adata.X)
    X_target = adata.obsm.get('X_target', None)
    X_covar = adata.obsm.get('X_covar', None)
    
    print(f'X: {X.shape}')
    print(f'X_target: {X_target.shape if X_target is not None else None}')
    print(f'X_covar: {X_covar.shape if X_covar is not None else None}')

## 评估指标

- Top-20 差异基因 MSE
- Pearson 相关系数
- 表达变化方向错误率
- Precision@10
- 遗传相互作用 (R²)
- ARI / NMI